In [ ]:
#!/usr/bin/env python3
"""
v5 G3 v4 — FULL-SPACE EXACT LINEAR FDT: does the rank-50 PCA bottleneck hide a real signal?
Brian, 2026-07-20. Adjudicates ONE specific objection: that v2/v3 scored the model inside a
top-50 variance subspace (expl.var 0.134) that may DISCARD the low-variance directions a
specific perturbation response lives in. "Shining the light in the wrong subspace."

THE KEY REALIZATION
-------------------
The linear-FDT prediction for knocking down gene g is EXACTLY the control covariance column of g:
    pred_j = cov(x_g, x_j)          (Gaussian conditional / linear response; direction = Sigma_{.g})
This needs NO PCA. It is one pass over the full 18,080-gene control matrix. So the linear arm can
be evaluated in the FULL space where nothing is truncated. If the signal was being destroyed by the
rank-50 bottleneck, it reappears here. If it does not, the bottleneck is exonerated and the premise
is dead for a deeper reason (response genes do not co-fluctuate with g at baseline).

This ONLY rescues the linear arm — the C3 nonlinear term (18,080^3) genuinely cannot be done in
full space. But linear is the load-bearing arm: it is what ties the wrong-gene control and triggers
the KILL. If full-space linear is alive, the whole design reopens.

CONTROLS (unchanged discipline)
-------------------------------
- A/B cell split: calibrate/stratify on A, EVALUATE on B only.
- Wrong-gene specificity control: predict with a RANDOM other gene's covariance column.
- Both DES@K and correlation, resistant stratum, plus per-perturbation noise floor r_p.
- Covariance computed on CONTROL CELLS ONLY. log1p (raw counts verified).
- Everything full-space, exact, no dimensionality reduction anywhere.

PRE-REGISTERED (locked before running)
--------------------------------------
Primary = full-space linear DES@K vs wrong-gene DES@K, resistant stratum.
  REOPEN : full-space linear beats wrong-gene by >0.05 on DES. The rank-50 bottleneck WAS
           hiding the signal. v2/v3 KILL is overturned; rebuild in full space; solve the
           nonlinear reduction as a second step.
  CONFIRM-KILL : full-space linear ties wrong-gene (<=0.05). The subspace is exonerated. FDT
           fails because response genes do not co-fluctuate with the target at baseline —
           a real non-equilibrium effect, not a measurement artifact. Design dead, airtight.
Report verbatim either way. No model may exceed the noise floor.
"""
import glob, sys
import numpy as np, pandas as pd, scanpy as sc, scipy.sparse as sp

MIN_CELLS, N_WRONG, SEED = 200, 3, 0
PERT_COL, CONTROL = "target_gene", "non-targeting"

def find_h5ad():
    for pat in ["/kaggle/input/**/adata_Training.h5ad", "**/adata_Training.h5ad"]:
        h = glob.glob(pat, recursive=True)
        if h: return h[0]
    sys.exit("FATAL: adata_Training.h5ad not found.")
def dense(x): return x.toarray() if sp.issparse(x) else np.asarray(x)
def corr(a,b):
    a=np.asarray(a,float)-np.asarray(a,float).mean(); b=np.asarray(b,float)-np.asarray(b,float).mean()
    d=np.sqrt((a*a).sum()*(b*b).sum()); return float((a*b).sum()/d) if d>0 else np.nan
def norm(M,t=1e4):
    s=M.sum(1,keepdims=True); s[s==0]=1; return np.log1p(M/s*t)
def des_k(pred,truth):
    at=np.abs(truth); mad=np.median(np.abs(at-np.median(at)))*1.4826+1e-9
    K=int(np.clip((at>np.median(at)+3*mad).sum(),20,200))
    return len(set(np.argsort(-at)[:K]) & set(np.argsort(-np.abs(pred))[:K]))/K

# ------------------------------------------------------------------ load
path=find_h5ad(); print(f"loading {path}")
ad=sc.read_h5ad(path); print(f"  {ad.n_obs:,} cells x {ad.n_vars:,} genes")
Xs=dense(ad.X[:200]); print(f"  VERIFY .X integral={np.allclose(Xs,np.round(Xs))} max={Xs.max():.1f}")

ctrl=dense(ad[ad.obs[PERT_COL]==CONTROL].X); ctrlN=norm(ctrl)
ctrl_mean=ctrlN.mean(0); Xc=ctrlN-ctrl_mean          # centered control, FULL space
Nc=Xc.shape[0]
print(f"  control cells: {Nc:,}  (full-space covariance columns computed exactly, NO PCA)")
var_g=(Xc*Xc).sum(0)/(Nc-1) + 1e-12                  # per-gene control variance

def cov_col(g):
    """Exact full-space covariance column of gene g: cov(x_g, x_.) over control cells."""
    return (Xc[:,g] @ Xc)/(Nc-1)                      # length-18080, exact

# ------------------------------------------------------------------ evaluate
gix={g:i for i,g in enumerate(ad.var_names)}; rng=np.random.default_rng(SEED)
recs=[]
for p in [q for q in ad.obs[PERT_COL].unique() if q!=CONTROL]:
    if p not in gix: continue
    M=dense(ad[ad.obs[PERT_COL]==p].X)
    if M.shape[0]<MIN_CELLS: continue
    Mn=norm(M); gi=gix[p]; idx=rng.permutation(Mn.shape[0]); h=len(idx)//2
    dxA=Mn[idx[:h]].mean(0)-ctrl_mean; dxB=Mn[idx[h:]].mean(0)-ctrl_mean
    m=np.ones(ad.n_vars,bool); m[gi]=False
    # linear FDT, exact full space: conditional-mean direction = cov_col(g)/var_g[g], scaled to
    # match observed knockdown at g in the calibration half (non-circular: uses dxA, scores dxB)
    pred = cov_col(gi)/var_g[gi] * dxA[gi]
    wc,wd=[],[]
    for wg in rng.choice([i for i in range(ad.n_vars) if i!=gi], N_WRONG, replace=False):
        wp = cov_col(wg)/var_g[wg] * dxA[gi]
        wc.append(corr(wp[m],dxB[m])); wd.append(des_k(wp[m],dxB[m]))
    recs.append(dict(pert=p,n=int(M.shape[0]),
                     r_p=corr(dxA[m],dxB[m]), r_p_des=des_k(dxA[m],dxB[m]),
                     effect=float(np.sqrt((dxA[m]**2).mean())),
                     corr_lin=corr(pred[m],dxB[m]), des_lin=des_k(pred[m],dxB[m]),
                     corr_wrong=float(np.nanmean(wc)), des_wrong=float(np.nanmean(wd))))
    print(f"  {p:10s} n={M.shape[0]:5d} DES full-lin={recs[-1]['des_lin']:.2f} "
          f"wrong={recs[-1]['des_wrong']:.2f} floor={recs[-1]['r_p_des']:.2f}")

df=pd.DataFrame(recs); df.to_csv("v5g3_fullspace_linear_v4.csv",index=False)
hi=df[df.effect>=df.effect.median()]
def line(n,a,b): print(f"  {n:20s} {a:+.4f}   (resistant {b:+.4f})")
print(f"\n=== FULL-SPACE LINEAR FDT  (n={len(df)}; resistant n={len(hi)}) ===")
print("--- DES@K (benchmark metric, movers only) ---")
line("noise floor r_p", df.r_p_des.mean(), hi.r_p_des.mean())
line("wrong-gene", df.des_wrong.mean(), hi.des_wrong.mean())
line("FULL-SPACE linear", df.des_lin.mean(), hi.des_lin.mean())
print("--- correlation (background-dominated) ---")
line("noise floor r_p", df.r_p.mean(), hi.r_p.mean())
line("wrong-gene", df.corr_wrong.mean(), hi.corr_wrong.mean())
line("FULL-SPACE linear", df.corr_lin.mean(), hi.corr_lin.mean())

d = hi.des_lin.mean() - hi.des_wrong.mean()
print(f"\n=== PRE-REGISTERED VERDICT ===")
print(f"  full-space linear DES {hi.des_lin.mean():.4f}  -  wrong-gene {hi.des_wrong.mean():.4f}  =  {d:+.4f}")
if d > 0.05:
    print("  REOPEN - full-space linear beats wrong-gene on DES. The rank-50 PCA bottleneck")
    print("           WAS hiding the signal. v2/v3 KILL is overturned. Rebuild full-space;")
    print("           solve the nonlinear reduction next. Brian was shining the wrong light.")
else:
    print("  CONFIRM-KILL - full-space linear ties wrong-gene even with NO truncation. The")
    print("           subspace is exonerated. FDT fails because response genes do not")
    print("           co-fluctuate with the target at baseline. Design dead, now airtight.")
print("\nCompare directly to v3 rank-50 DES (linear resistant 0.1066, wrong 0.1218).")
print("Copy verbatim into STATE.md.")
